# Köln Cool City Lab: Ölçümden yoruma

**Soru:** Katran, kum, çim ve çakıl kaplı kutuların iç sıcaklıkları **aynı ölçüm saatlerinde** ne kadar farklıydı?

Bu derste sensör serilerini okuyacak, eksik veriyi tanıyacak, ortak saatleri seçecek ve farkları hesaplayacaksın. Hücreleri yukarıdan aşağı çalıştır. Varsayılan mod **sentetik alıştırma**; sonuçları Köln ölçümü olarak yorumlama.

**Kurulum:** Depo kökünde `python -m pip install -r requirements.txt jupyterlab` ve `jupyter lab` komutlarını çalıştır. Gerçek veri için [README](README.md) içindeki iki orijinal dosyayı `data/` klasörüne indir ve aşağıdaki seçeneği `True` yap. Dosyalar yoksa sentetik moda sessizce geçilmez.

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

from analyze import START, END, SURFACES, paired_hours, plot, read_boxes, read_weather, summarize

USE_REAL_DATA = False

## 1. Veriyi tanı ve temizle

Üniversitenin [deney günlüğü](https://geographie.uni-koeln.de/en/forschung/arbeitsgruppen/hydrogeographie-und-klimatologie/projektuebersicht/projekte/projektseiten/pulchra/the-cool-city-lab) kutu–sensör eşleşmesini ve deney düzenindeki değişiklikleri belgeliyor. Gerçek analiz, çimin geri konmasından sonraki **7 Haziran–13 Temmuz 2021** dönemini kapsıyor; 14 Temmuz dahil değil.

`read_boxes` geçersiz tarih/sıcaklıkları ayıklar ve bir sensörde aynı zaman damgasına sahip geçerli kayıtları ortalar. Böylece tekrarlar kapsama sayısını şişirmez. Aşağıdaki sentetik örnekte ikinci saatte çim kutusundan üç kayıt çıkarıyoruz. Bunun eşleştirmeye etkisini tahmin et.

In [2]:
if USE_REAL_DATA:
    mode = "gercek"
    series, quality = read_boxes(Path("data/Cool_City_Lab_Temperaturen_Boxen.xlsx"))
    weather = read_weather(Path("data/meteo_CoolCityLab.csv"))
    display(pd.DataFrame(quality).T)
else:
    mode = "sentetik"
    time = pd.date_range(START, periods=24, freq="5min")
    series = {
        name: pd.Series(20.0 + offset, index=time)
        for name, offset in {"sand": 0.8, "grass": 0.0, "gravel": 0.4, "tar": 2.0}.items()
    }
    series["grass"] = series["grass"].drop(time[12:15])

display(pd.DataFrame.from_dict(SURFACES, orient="index", columns=["Örtü"]).rename_axis("Sensör"))
display(pd.concat(series, axis=1, sort=True).loc[START:].head(4))

,Örtü
Sensör,
15,sand
16,grass
18,gravel
19,tar


,sand,grass,gravel,tar
2021-06-07 00:00:00,20.8,20.0,20.4,22.0
2021-06-07 00:05:00,20.8,20.0,20.4,22.0
2021-06-07 00:10:00,20.8,20.0,20.4,22.0
2021-06-07 00:15:00,20.8,20.0,20.4,22.0


## 2. Ortak saatleri seç

Önce her kutunun saatlik ortalamasını ayrı hesaplıyoruz. Sonra **dört kutunun da** en az 10 geçerli ölçümü olan saatleri tutuyoruz. 5 dakikalık düzenli aralıkta bir saatte 12 kayıt beklenir. Eşik tam 12 şartından biraz daha esnek; eksik değerleri tahmin ederek doldurmuyoruz.

Kutuların saat dilimi belgelenmemiş. Bu yüzden kutuları kaydedilen saatlerine göre eşleştiriyoruz. Meteoroloji istasyonunun UTC serisini kutularla saat saat birleştirmiyoruz.

In [3]:
eligible = {surface: s.loc[(s.index >= START) & (s.index < END)] for surface, s in series.items()}
counts = pd.DataFrame({name: values.resample("h").count() for name, values in eligible.items()})
display(counts.head())
pairs = paired_hours(series, 10)
print(f"Ortak saat sayısı: {len(pairs)}")
display(pairs.head())

,sand,grass,gravel,tar
2021-06-07 00:00:00,12,12,12,12
2021-06-07 01:00:00,12,9,12,12


Ortak saat sayısı: 1


,sand,grass,gravel,tar
2021-06-07,20.8,20.0,20.4,22.0


## 3. Farkı hesapla ve yorumla

Her ortak saatte **katran eksi karşılaştırma kutusu** hesabını yap. Örneğin 22 °C − 20 °C = +2 °C. Artı değer, o saatte katran kutusunun daha sıcak olduğunu gösterir. Önce farkı almak, karşılaştırmanın aynı saatler üzerinden yapılmasını sağlar.

Ortalama fark tipik büyüklüğe dair bir özet verir; medyan uç değerlere daha az duyarlıdır. Katranın daha sıcak olduğu saat oranı ise farkın ne kadar sık pozitif olduğunu söyler.

In [4]:
results = pd.DataFrame(summarize(pairs)).T
display(results.round(3))
if not USE_REAL_DATA:
    assert results.loc["grass", "mean_c"] == 2.0

,mean_c,median_c,p10_c,p90_c,tar_warmer_fraction
sand,1.2,1.2,1.2,1.2,1.0
grass,2.0,2.0,2.0,2.0,1.0
gravel,1.6,1.6,1.6,1.6,1.0


## 4. Sonuç kapsama eşiğine duyarlı mı?

Aynı hesabı en az **12 kayıt** koşuluyla tekrarla. Daha sıkı bir eşik karşılaştırılabilirliği artırabilir ama veri kaybına yol açar. Sonucu ve kalan saat sayısını birlikte değerlendir.

In [5]:
strict = paired_hours(series, 12)
print(f"10 kayıt: {len(pairs)} saat | 12 kayıt: {len(strict)} saat")
if not strict.empty:
    strict_results = pd.DataFrame(summarize(strict)).T
    display(pd.DataFrame({"mean_min10_C": results["mean_c"], "mean_min12_C": strict_results["mean_c"]}).round(3))

10 kayıt: 1 saat | 12 kayıt: 1 saat


,mean_min10_C,mean_min12_C
sand,1.2,1.2
grass,2.0,2.0
gravel,1.6,1.6


## 5. Gerçek sonuç grafiğini oku

- **A:** Günlük kutu ortalamaları, en az 18 ortak saatin bulunduğu günler.
- **B:** Kayıt saatine göre ortalama kutu sıcaklığı. Kutular arasındaki fark günün hangi bölümünde açılıyor?
- **C:** İstasyonun UTC günlerine göre ayrı hava sıcaklığı özeti. A ve C'yi saatlik olarak eşleşmiş sayma.

Sentetik modda depodaki **gerçek ölçüm grafiği** gösterilir; yukarıdaki sentetik tablolardan üretilmemiştir. Gerçek modda kendi çıktın yeniden hesaplanır.

![Cool City Lab gerçek ölçüm grafiği](docs/overview.png)

In [6]:
if USE_REAL_DATA:
    expected_days = pd.date_range(START, END, freq="D", inclusive="left", tz="UTC")
    bins = weather["airtemp"].resample("D")
    daily_weather = bins.mean().where(bins.count() >= 216).reindex(expected_days)
    out = Path("outputs/tutorial-real")
    out.mkdir(parents=True, exist_ok=True)
    plot(pairs, daily_weather.to_frame(), out)
    display(Image(filename=str(out / "overview.png"), width=850))

## Kendini dene

1. Sentetik örnekte ikinci saat neden elendi? Çimden yalnızca iki kayıt çıkarırsan ne değişir?
2. Gerçek modda katran–çim farkının ortalaması ve medyanı neden aynı olmak zorunda değil?
3. Tek kutu/örtü ile neden “çim Köln’ü şu kadar soğutur” diyemiyoruz?

**Kontrol:** İlk durumda çimde 9 kayıt kalır ve saat elenir; iki kayıt çıkarırsan 10 eşiğini geçer, 12 eşiğini geçmez. Ortalama ile medyan fark dağılımının farklı özelliklerini özetler. Son olarak bu ölçümler kutu içi sıcaklıklardır; saatler bağımsız deney tekrarları değildir ve kutu konumu, sulama ve başka koşullar farkı etkileyebilir.